In [14]:
!pip install -q \
    google-cloud-bigquery \
    pandas \
    pandas-gbq \
    pyarrow \
    db-dtypes

In [15]:
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# ============================================================
# Project configuration
# ============================================================

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path(
    "/content/drive/MyDrive/ENARES_2024_PROJECT"
)

LOG_DIR = (
    ROOT_DRIVE
    / "05Resultados"
    / "logs"
    / "stage03"
)

SQL_DIR = ROOT_DRIVE / "02SQL"

DOCS_DIR = ROOT_DRIVE / "docs"

for directory in [
    LOG_DIR,
    SQL_DIR,
    DOCS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RUN_UTC = datetime.now(
    timezone.utc
).isoformat()

print("PROJECT_ID:", PROJECT_ID)
print("ROOT_DRIVE:", ROOT_DRIVE)
print("RUN_UTC:", RUN_UTC)

PROJECT_ID: enares-2024-crs04
ROOT_DRIVE: /content/drive/MyDrive/ENARES_2024_PROJECT
RUN_UTC: 2026-07-17T04:23:44.041067+00:00


In [17]:
# ============================================================
# BigQuery client and target analytical table
# ============================================================

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

print("Target table:", A)

Target table: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents


In [18]:
# ============================================================
# Validate BigQuery connection and analytical table
# ============================================================

try:
    table = client.get_table(A)
except Exception as exc:
    raise RuntimeError(
        "The analytical table could not be opened. "
        "Run the previous Stage 03 notebooks first."
    ) from exc

existing_columns = {
    field.name
    for field in table.schema
}

print("BigQuery connection: OK")
print("Rows:", table.num_rows)
print("Columns:", len(existing_columns))

if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"The analytical table contains "
        f"{table.num_rows:,} rows; "
        f"{EXPECTED_ROWS:,} were expected."
    )

minimum_required = [
    "C3P114_VS",
    "C3P120B",
    "C3P121",
    "C3P122",
    "C3P123",
    "C3P302_4",
    "C4P248_1",
]

missing_minimum = sorted(
    set(minimum_required)
    - existing_columns
)

if missing_minimum:
    raise RuntimeError(
        "The analytical table does not contain the "
        "required CRS04 source variables. Missing: "
        + ", ".join(missing_minimum)
    )

print(
    "Prerequisite validation passed: "
    "analytical table exists, row count is correct, "
    "and CRS04 source columns are available."
)

BigQuery connection: OK
Rows: 18807
Columns: 1355
Prerequisite validation passed: analytical table exists, row count is correct, and CRS04 source columns are available.


In [19]:
# ============================================================
# STAGE 03 — Ficha de riesgo de desprotección familiar
# Traducción del bloque CRS04: adolescentes de 12 a 17 años
#
# Archivo fuente:
# 14_CRS03_Ficha_Riesgo_Desproteccion_NN_9a11.sps
#
# Salidas:
# - 15 señales rd12_*
# - rd12_senales_total
# - rd12_riesgo_1mas
# - rd12_riesgo_2mas
#
# BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE ni ALTER TABLE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Definir variables de salida
# ------------------------------------------------------------

rd12_signal_columns = [
    "rd12_114_vive_solo_12a14",
    "rd12_120b_solo_noche_dia_diario",
    "rd12_121_sin_comer",
    "rd12_122_no_ir_colegio",
    "rd12_123_peleas_frecuentes",
    "rd12_vp201_1a5",
    "rd12_vp201_6a9",
    "rd12_vf205_1a2",
    "rd12_vf205_3a6",
    "rd12_218a_1a5",
    "rd12_218b_miedo_dormir_frec",
    "rd12_218c_1a5",
    "rd12_247_consec_1a5",
    "rd12_248_vs_familia",
    "rd12_302_4_adolescente_provee",
]

rd12_summary_columns = [
    "rd12_senales_total",
    "rd12_riesgo_1mas",
    "rd12_riesgo_2mas",
]

rd12_output_columns = (
    rd12_signal_columns
    + rd12_summary_columns
)


# ------------------------------------------------------------
# 2. Definir variables fuente requeridas
# ------------------------------------------------------------

required_rd12_sources = [
    # Negligencia y entorno familiar
    "EDAD",
    "C3P103EDAD",
    "C3P114_VS",
    "C3P120B",
    "C3P120C",
    "C3P120D",
    "C3P121",
    "C3P122",
    "C3P123",
    "C3P124",

    # Condiciones generales de violencia psicológica
    "C3P203",
    "C3P204",

    # Condiciones generales de violencia física
    "C3P207",
    "C3P208",

    # Señales 218A, 218B y 218C
    "C3P216B_2",
    "C3P216B_2C",

    # Trabajo/provisión del hogar
    "C3P302_4",
]

# Violencia psicológica 201: nueve situaciones
for i in range(1, 10):
    required_rd12_sources.extend([
        f"C3P201_{i}",
        f"C3P201A_{i}",
        f"C3P201B_{i}",
        f"C3P201C_{i}",
        f"C3P201D_{i}",
    ])

# Violencia física 205: seis situaciones
for i in range(1, 7):
    required_rd12_sources.extend([
        f"C3P205_{i}",
        f"C3P205A_{i}",
        f"C3P205B_{i}",
        f"C3P205C_{i}",
        f"C3P205D_{i}",
    ])

# Preguntas 218A y 218C
for i in range(1, 6):
    required_rd12_sources.extend([
        f"C3P216A_{i}",
        f"C3P216C_{i}",
    ])

# Consecuencias físicas 247
for i in range(1, 6):
    required_rd12_sources.append(
        f"C3P243_{i}"
    )

# Violencia sexual 248:
# 16 formas y agresores familiares con códigos 1–17
for item in range(1, 17):
    required_rd12_sources.append(
        f"C4P248_{item}"
    )

    for aggressor in range(1, 18):
        required_rd12_sources.append(
            f"C4P248A_{aggressor}_{item}"
        )


# ------------------------------------------------------------
# 3. Verificar esquema
# ------------------------------------------------------------

table_before = client.get_table(A)

existing_columns = {
    field.name
    for field in table_before.schema
}

missing_rd12_sources = sorted(
    set(required_rd12_sources) - existing_columns
)

if missing_rd12_sources:
    print(
        f"Variables fuente faltantes: "
        f"{len(missing_rd12_sources)}"
    )

    for variable in missing_rd12_sources:
        print(f" - {variable}")

    raise RuntimeError(
        "No se puede construir el bloque rd12. "
        "Revisa el merge de los capítulos CRS04."
    )

if table_before.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La tabla contiene {table_before.num_rows} filas; "
        f"se esperaban {EXPECTED_ROWS}."
    )

print(
    "Prerequisitos aprobados: "
    f"{len(set(required_rd12_sources))} variables fuente "
    f"y {table_before.num_rows:,} filas."
)


# ------------------------------------------------------------
# 4. Preparar reejecución segura
# ------------------------------------------------------------

columns_to_replace = [
    column
    for column in rd12_output_columns
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(
            f"`{column}`"
            for column in columns_to_replace
        )
        + ")"
    )
else:
    source_select = "*"


# ------------------------------------------------------------
# 5. Violencia psicológica 201
# ------------------------------------------------------------

def family_origin_condition(prefix, item):
    """
    Traduce el criterio SPSS:

    item = 1
    AND pregunta de ocurrencia = 1
    AND frecuencia = 3
    AND (
        A IN (1,2,3,4) AND B = 1
        OR
        A BETWEEN 5 AND 17
        AND B = 1 AND C = 1 AND D = 1
    )
    """

    return f"""
    (
      `{prefix}_{item}` = 1
      AND C3P203 = 1
      AND C3P204 = 3
      AND (
        (
          `{prefix}A_{item}` IN (1, 2, 3, 4)
          AND `{prefix}B_{item}` = 1
        )
        OR
        (
          `{prefix}A_{item}` BETWEEN 5 AND 17
          AND `{prefix}B_{item}` = 1
          AND `{prefix}C_{item}` = 1
          AND `{prefix}D_{item}` = 1
        )
      )
    )
    """


vp201_conditions = {
    i: family_origin_condition("C3P201", i)
    for i in range(1, 10)
}

vp201_1a5_condition = "\nOR\n".join(
    vp201_conditions[i]
    for i in range(1, 6)
)

vp201_6a9_condition = "\nOR\n".join(
    vp201_conditions[i]
    for i in range(6, 10)
)


# ------------------------------------------------------------
# 6. Violencia física 205
# ------------------------------------------------------------

def physical_family_condition(item):
    return f"""
    (
      `C3P205_{item}` = 1
      AND C3P207 = 1
      AND C3P208 = 3
      AND (
        (
          `C3P205A_{item}` IN (1, 2, 3, 4)
          AND `C3P205B_{item}` = 1
        )
        OR
        (
          `C3P205A_{item}` BETWEEN 5 AND 17
          AND `C3P205B_{item}` = 1
          AND `C3P205C_{item}` = 1
          AND `C3P205D_{item}` = 1
        )
      )
    )
    """


vf205_conditions = {
    i: physical_family_condition(i)
    for i in range(1, 7)
}

vf205_1a2_condition = "\nOR\n".join(
    vf205_conditions[i]
    for i in range(1, 3)
)

vf205_3a6_condition = "\nOR\n".join(
    vf205_conditions[i]
    for i in range(3, 7)
)


# ------------------------------------------------------------
# 7. Violencia sexual 248 por familiar
# ------------------------------------------------------------

vs248_item_conditions = []

for item in range(1, 17):

    family_aggressors = " OR ".join(
        f"`C4P248A_{aggressor}_{item}` = 1"
        for aggressor in range(1, 18)
    )

    vs248_item_conditions.append(f"""
    (
      `C4P248_{item}` = 1
      AND (
        {family_aggressors}
      )
    )
    """)

vs248_family_condition = "\nOR\n".join(
    vs248_item_conditions
)


# ------------------------------------------------------------
# 8. Crear las 15 señales
# ------------------------------------------------------------

signals_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH rd12_signals AS (

  SELECT
    {source_select},

    -- 114: vive sola/o y tiene entre 12 y 14 años
    CASE
      WHEN C3P114_VS = 1
       AND (
         EDAD BETWEEN 12 AND 14
         OR C3P103EDAD BETWEEN 12 AND 14
       )
      THEN 1
      ELSE 0
    END AS rd12_114_vive_solo_12a14,

    -- 120B/C/D: permanece sin adulto toda la noche
    -- o todo el día, todos los días
    CASE
      WHEN C3P120B = 1
       AND C3P120C IN (3, 4)
       AND C3P120D = 1
      THEN 1
      ELSE 0
    END AS rd12_120b_solo_noche_dia_diario,

    -- 121: le dejaron sin comer por un día o más
    CASE
      WHEN C3P121 = 1
      THEN 1
      ELSE 0
    END AS rd12_121_sin_comer,

    -- 122: le pidieron no ir al colegio para ayudar
    CASE
      WHEN C3P122 = 1
      THEN 1
      ELSE 0
    END AS rd12_122_no_ir_colegio,

    -- 123/124: peleas o discusiones frecuentes
    CASE
      WHEN C3P123 = 1
       AND C3P124 = 3
      THEN 1
      ELSE 0
    END AS rd12_123_peleas_frecuentes,

    -- Violencia psicológica 201, situaciones 1–5
    CASE
      WHEN (
        {vp201_1a5_condition}
      )
      THEN 1
      ELSE 0
    END AS rd12_vp201_1a5,

    -- Violencia psicológica 201, situaciones 6–9
    CASE
      WHEN (
        {vp201_6a9_condition}
      )
      THEN 1
      ELSE 0
    END AS rd12_vp201_6a9,

    -- Violencia física 205, situaciones 1–2
    CASE
      WHEN (
        {vf205_1a2_condition}
      )
      THEN 1
      ELSE 0
    END AS rd12_vf205_1a2,

    -- Violencia física 205, situaciones 3–6
    CASE
      WHEN (
        {vf205_3a6_condition}
      )
      THEN 1
      ELSE 0
    END AS rd12_vf205_3a6,

    -- 218A: al menos una situación 1–5
    CASE
      WHEN
        C3P216A_1 = 1
        OR C3P216A_2 = 1
        OR C3P216A_3 = 1
        OR C3P216A_4 = 1
        OR C3P216A_5 = 1
      THEN 1
      ELSE 0
    END AS rd12_218a_1a5,

    -- 218B.2: no puede dormir por miedo,
    -- siempre o casi siempre
    CASE
      WHEN C3P216B_2 = 1
       AND C3P216B_2C = 3
      THEN 1
      ELSE 0
    END AS rd12_218b_miedo_dormir_frec,

    -- 218C: persona cuidadora pidió realizar
    -- alguna situación 1–5
    CASE
      WHEN
        C3P216C_1 = 1
        OR C3P216C_2 = 1
        OR C3P216C_3 = 1
        OR C3P216C_4 = 1
        OR C3P216C_5 = 1
      THEN 1
      ELSE 0
    END AS rd12_218c_1a5,

    -- 247: consecuencias físicas 1–5
    -- Variables reales en la base: C3P243_1 a C3P243_5
    CASE
      WHEN
        C3P243_1 = 1
        OR C3P243_2 = 1
        OR C3P243_3 = 1
        OR C3P243_4 = 1
        OR C3P243_5 = 1
      THEN 1
      ELSE 0
    END AS rd12_247_consec_1a5,

    -- 248: violencia sexual por familiar
    -- o integrante de la familia de origen
    -- Códigos de agresor 1–17; sin restricción temporal
    CASE
      WHEN (
        {vs248_family_condition}
      )
      THEN 1
      ELSE 0
    END AS rd12_248_vs_familia,

    -- 302.4: adolescente realiza con mayor frecuencia
    -- la provisión de dinero o gastos del hogar
    CASE
      WHEN C3P302_4 = 1
      THEN 1
      ELSE 0
    END AS rd12_302_4_adolescente_provee

  FROM `{A}`
),

rd12_totals AS (

  SELECT
    *,

    (
      rd12_114_vive_solo_12a14
      + rd12_120b_solo_noche_dia_diario
      + rd12_121_sin_comer
      + rd12_122_no_ir_colegio
      + rd12_123_peleas_frecuentes
      + rd12_vp201_1a5
      + rd12_vp201_6a9
      + rd12_vf205_1a2
      + rd12_vf205_3a6
      + rd12_218a_1a5
      + rd12_218b_miedo_dormir_frec
      + rd12_218c_1a5
      + rd12_247_consec_1a5
      + rd12_248_vs_familia
      + rd12_302_4_adolescente_provee
    ) AS rd12_senales_total

  FROM rd12_signals
)

SELECT
  *,

  CASE
    WHEN rd12_senales_total >= 1 THEN 1
    ELSE 0
  END AS rd12_riesgo_1mas,

  CASE
    WHEN rd12_senales_total > 1 THEN 1
    ELSE 0
  END AS rd12_riesgo_2mas

FROM rd12_totals
"""

(
    SQL_DIR
    / "stage3_ficha_riesgo_desproteccion_crs04.sql"
).write_text(
    signals_sql,
    encoding="utf-8",
)

print(signals_sql)

client.query(signals_sql).result()

print(
    "Bloque CRS04 de riesgo de desprotección "
    "creado correctamente."
)


# ------------------------------------------------------------
# 9. Validación general
# ------------------------------------------------------------

signal_validation_parts = []

for variable in rd12_signal_columns + [
    "rd12_riesgo_1mas",
    "rd12_riesgo_2mas",
]:

    query = f"""
    SELECT
      '{variable}' AS variable,
      COUNT(*) AS total_rows,

      COUNTIF(
        `{variable}` IS NULL
        OR `{variable}` NOT IN (0, 1)
      ) AS invalid_values,

      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values

    FROM `{A}`
    """

    signal_validation_parts.append(
        client.query(query)
        .result()
        .to_dataframe()
    )

rd12_domain_validation = pd.concat(
    signal_validation_parts,
    ignore_index=True,
)

rd12_domain_validation.to_csv(
    LOG_DIR
    / "stage3_ficha_riesgo_crs04_domain_validation.csv",
    index=False,
)

display(rd12_domain_validation)

if (
    rd12_domain_validation["total_rows"] != EXPECTED_ROWS
).any():
    raise RuntimeError(
        "Alguna variable rd12 alteró el universo analítico."
    )

if (
    rd12_domain_validation["invalid_values"] > 0
).any():
    raise RuntimeError(
        "Alguna variable binaria rd12 contiene "
        "valores fuera de 0/1."
    )


# ------------------------------------------------------------
# 10. Validar total y consistencia de indicadores
# ------------------------------------------------------------

rd12_consistency = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  MIN(rd12_senales_total) AS min_signals,
  MAX(rd12_senales_total) AS max_signals,

  COUNTIF(
    rd12_senales_total IS NULL
    OR rd12_senales_total < 0
    OR rd12_senales_total > 15
  ) AS invalid_signal_total,

  COUNTIF(
    rd12_riesgo_1mas !=
    CASE
      WHEN rd12_senales_total >= 1 THEN 1
      ELSE 0
    END
  ) AS bad_riesgo_1mas,

  COUNTIF(
    rd12_riesgo_2mas !=
    CASE
      WHEN rd12_senales_total > 1 THEN 1
      ELSE 0
    END
  ) AS bad_riesgo_2mas

FROM `{A}`
""").result().to_dataframe()

rd12_consistency.to_csv(
    LOG_DIR
    / "stage3_ficha_riesgo_crs04_consistency_validation.csv",
    index=False,
)

display(rd12_consistency)

r = rd12_consistency.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "El bloque rd12 alteró el universo analítico."
    )

if r["invalid_signal_total"] > 0:
    raise RuntimeError(
        "rd12_senales_total contiene valores fuera de 0–15."
    )

if (
    r["bad_riesgo_1mas"] > 0
    or r["bad_riesgo_2mas"] > 0
):
    raise RuntimeError(
        "Los indicadores resumen rd12 son inconsistentes."
    )

print(
    "Validación aprobada: "
    "18,807 filas, señales binarias, total 0–15 "
    "e indicadores resumen consistentes."
)


# ------------------------------------------------------------
# 11. Distribución del total de señales
# ------------------------------------------------------------

rd12_distribution = client.query(f"""
SELECT
  rd12_senales_total,
  COUNT(*) AS n,

  ROUND(
    100 * SAFE_DIVIDE(
      COUNT(*),
      SUM(COUNT(*)) OVER ()
    ),
    4
  ) AS percent

FROM `{A}`

GROUP BY rd12_senales_total
ORDER BY rd12_senales_total
""").result().to_dataframe()

rd12_distribution.to_csv(
    LOG_DIR
    / "stage3_ficha_riesgo_crs04_distribution.csv",
    index=False,
)

display(rd12_distribution)

Prerequisitos aprobados: 395 variables fuente y 18,807 filas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH rd12_signals AS (

  SELECT
    * EXCEPT(`rd12_114_vive_solo_12a14`, `rd12_120b_solo_noche_dia_diario`, `rd12_121_sin_comer`, `rd12_122_no_ir_colegio`, `rd12_123_peleas_frecuentes`, `rd12_vp201_1a5`, `rd12_vp201_6a9`, `rd12_vf205_1a2`, `rd12_vf205_3a6`, `rd12_218a_1a5`, `rd12_218b_miedo_dormir_frec`, `rd12_218c_1a5`, `rd12_247_consec_1a5`, `rd12_248_vs_familia`, `rd12_302_4_adolescente_provee`, `rd12_senales_total`, `rd12_riesgo_1mas`, `rd12_riesgo_2mas`),

    -- 114: vive sola/o y tiene entre 12 y 14 años
    CASE
      WHEN C3P114_VS = 1
       AND (
         EDAD BETWEEN 12 AND 14
         OR C3P103EDAD BETWEEN 12 AND 14
       )
      THEN 1
      ELSE 0
    END AS rd12_114_vive_solo_12a14,

    -- 120B/C/D: permanece sin adulto toda la noche
    -- o todo el día, todos los días
    CASE
      WHEN C3P120B = 1


,variable,total_rows,invalid_values,zero_values,one_values
0,rd12_114_vive_solo_12a14,18807,0,18804,3
1,rd12_120b_solo_noche_dia_diario,18807,0,18773,34
2,rd12_121_sin_comer,18807,0,18484,323
3,rd12_122_no_ir_colegio,18807,0,17420,1387
4,rd12_123_peleas_frecuentes,18807,0,17995,812
5,rd12_vp201_1a5,18807,0,18092,715
6,rd12_vp201_6a9,18807,0,18345,462
7,rd12_vf205_1a2,18807,0,18599,208
8,rd12_vf205_3a6,18807,0,18582,225
9,rd12_218a_1a5,18807,0,13377,5430


,total_rows,min_signals,max_signals,invalid_signal_total,bad_riesgo_1mas,bad_riesgo_2mas
0,18807,0,12,0,0,0


Validación aprobada: 18,807 filas, señales binarias, total 0–15 e indicadores resumen consistentes.


,rd12_senales_total,n,percent
0,0,10322,54.8838
1,1,4855,25.8149
2,2,2046,10.8789
3,3,829,4.4079
4,4,350,1.8610
5,5,205,1.0900
6,6,101,0.5370
7,7,56,0.2978
8,8,29,0.1542
9,9,7,0.0372


In [20]:
# ============================================================
# 10b — Verificar dependencias previas
# ============================================================

required_10b = [
    "VS_12M",
    "rd12_114_vive_solo_12a14",
    "rd12_120b_solo_noche_dia_diario",
    "rd12_121_sin_comer",
    "rd12_122_no_ir_colegio",
    "rd12_302_4_adolescente_provee",
]

table_10b = client.get_table(A)

existing_columns = {
    field.name
    for field in table_10b.schema
}

missing_10b = sorted(
    set(required_10b) - existing_columns
)

if missing_10b:
    raise RuntimeError(
        "Todavía faltan variables para ejecutar 10b: "
        + ", ".join(missing_10b)
    )

if table_10b.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La tabla tiene {table_10b.num_rows:,} filas; "
        f"se esperaban {EXPECTED_ROWS:,}."
    )

print("Dependencias 10b aprobadas.")
print("Filas:", table_10b.num_rows)
print("Variables requeridas:", len(required_10b))

Dependencias 10b aprobadas.
Filas: 18807
Variables requeridas: 6


In [23]:
# ============================================================
# 10b — Crear rd12_idx_noviol y rd12_idx_noviol_cat
# Versión corregida y reejecutable
# ============================================================

required_sources = [
    "rd12_114_vive_solo_12a14",
    "rd12_120b_solo_noche_dia_diario",
    "rd12_121_sin_comer",
    "rd12_122_no_ir_colegio",
    "rd12_302_4_adolescente_provee",
]

table_before = client.get_table(A)

existing_columns = {
    field.name
    for field in table_before.schema
}

missing_sources = sorted(
    set(required_sources) - existing_columns
)

if missing_sources:
    raise RuntimeError(
        "No se pueden crear las variables 10b. "
        "Faltan señales fuente: "
        + ", ".join(missing_sources)
    )

outputs_10b = [
    "rd12_idx_noviol",
    "rd12_idx_noviol_cat",
]

columns_to_replace = [
    column
    for column in outputs_10b
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(
            f"`{column}`"
            for column in columns_to_replace
        )
        + ")"
    )
else:
    source_select = "*"

sql_10b_rd12 = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH base AS (
  SELECT
    {source_select},

    (
      COALESCE(rd12_114_vive_solo_12a14, 0)
      + COALESCE(rd12_120b_solo_noche_dia_diario, 0)
      + COALESCE(rd12_121_sin_comer, 0)
      + COALESCE(rd12_122_no_ir_colegio, 0)
      + COALESCE(rd12_302_4_adolescente_provee, 0)
    ) AS rd12_idx_noviol

  FROM `{A}`
)

SELECT
  *,

  CASE
    WHEN rd12_idx_noviol = 0 THEN 0
    WHEN rd12_idx_noviol = 1 THEN 1
    WHEN rd12_idx_noviol >= 2 THEN 2
    ELSE NULL
  END AS rd12_idx_noviol_cat

FROM base
"""

print(sql_10b_rd12)

query_job = client.query(
    sql_10b_rd12,
    location=LOCATION,
)

query_job.result()

print("CREATE OR REPLACE TABLE completed.")


CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH base AS (
  SELECT
    *,

    (
      COALESCE(rd12_114_vive_solo_12a14, 0)
      + COALESCE(rd12_120b_solo_noche_dia_diario, 0)
      + COALESCE(rd12_121_sin_comer, 0)
      + COALESCE(rd12_122_no_ir_colegio, 0)
      + COALESCE(rd12_302_4_adolescente_provee, 0)
    ) AS rd12_idx_noviol

  FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`
)

SELECT
  *,

  CASE
    WHEN rd12_idx_noviol = 0 THEN 0
    WHEN rd12_idx_noviol = 1 THEN 1
    WHEN rd12_idx_noviol >= 2 THEN 2
    ELSE NULL
  END AS rd12_idx_noviol_cat

FROM base

CREATE OR REPLACE TABLE completed.


In [24]:
# ============================================================
# Confirmar que BigQuery ya creó las columnas
# ============================================================

table_after = client.get_table(A)

columns_after = {
    field.name
    for field in table_after.schema
}

missing_outputs = sorted(
    set(outputs_10b) - columns_after
)

if missing_outputs:
    raise RuntimeError(
        "La consulta terminó, pero las columnas no aparecen: "
        + ", ".join(missing_outputs)
    )

print("Variables creadas correctamente:")
print(" - rd12_idx_noviol")
print(" - rd12_idx_noviol_cat")
print("Filas:", table_after.num_rows)

Variables creadas correctamente:
 - rd12_idx_noviol
 - rd12_idx_noviol_cat
Filas: 18807


In [25]:
# ============================================================
# 10b — Validación corregida
# ============================================================

validation_10b = client.query(
    f"""
    SELECT
      COUNT(*) AS total_rows,

      MIN(rd12_idx_noviol) AS min_idx,
      MAX(rd12_idx_noviol) AS max_idx,

      COUNTIF(
        rd12_idx_noviol IS NULL
        OR rd12_idx_noviol < 0
        OR rd12_idx_noviol > 5
      ) AS invalid_idx,

      COUNTIF(
        rd12_idx_noviol_cat IS NULL
        OR rd12_idx_noviol_cat NOT IN (0, 1, 2)
      ) AS invalid_category,

      COUNTIF(
        rd12_idx_noviol_cat !=
        CASE
          WHEN rd12_idx_noviol = 0 THEN 0
          WHEN rd12_idx_noviol = 1 THEN 1
          WHEN rd12_idx_noviol >= 2 THEN 2
        END
      ) AS inconsistent_category

    FROM `{A}`
    """,
    location=LOCATION,
).result().to_dataframe()

display(validation_10b)

validation_10b.to_csv(
    LOG_DIR / "stage3_10b_rd12_validation.csv",
    index=False,
)

r = validation_10b.iloc[0]

if int(r["total_rows"]) != EXPECTED_ROWS:
    raise RuntimeError(
        f"El bloque 10b dejó {int(r['total_rows']):,} filas; "
        f"se esperaban {EXPECTED_ROWS:,}."
    )

if int(r["invalid_idx"]) > 0:
    raise RuntimeError(
        "rd12_idx_noviol contiene valores inválidos."
    )

if int(r["invalid_category"]) > 0:
    raise RuntimeError(
        "rd12_idx_noviol_cat contiene valores inválidos."
    )

if int(r["inconsistent_category"]) > 0:
    raise RuntimeError(
        "La categoría no coincide con el índice."
    )

print(
    "Validación 10b aprobada: "
    "índice 0–5 y categoría 0/1/2."
)

,total_rows,min_idx,max_idx,invalid_idx,invalid_category,inconsistent_category
0,18807,0,3,0,0,0


Validación 10b aprobada: índice 0–5 y categoría 0/1/2.
